# Global Sensitivity C-revised — weighted class Diagnostics

Train-CV diagnostics only. Test·tuning·threshold 변경은 사용하지 않는다.


## 1. 입력과 25 Feature·weight 계약


In [ ]:
import json, os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
ROOT=Path(os.environ.get("KHUDA_PROJECT_ROOT",Path.cwd())).resolve()
while not (ROOT/"code").is_dir():
    if ROOT.parent==ROOT: raise RuntimeError("저장소 안에서 실행하세요.")
    ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
if "code" in sys.modules and not hasattr(sys.modules["code"],"__path__"): del sys.modules["code"]
from code.evaluation.evaluate import calculate_binary_metrics
from code.evaluation.sensitivity_diagnostics import cv_permutation_importance, normalize_oof_columns, paired_sampid_bootstrap_f1, subgroup_oof_metrics
from code.model.locked_sensitivity import load_stage_3_5_locked_params
from code.pipeline.audit import attach_person_period_column, calculate_train_sample_weight, load_selected_feature_names
from code.pipeline.saved_results import load_saved_global_train
RESULT_ROOT=ROOT/"data"/"result"/"baseline_42features"; STAGE35=RESULT_ROOT/"modeling"/"stage_3_5"
DATASET=RESULT_ROOT/"datasets"/"global_dataset.parquet"; SPLIT=RESULT_ROOT/"splits"/"split_ids.csv"; FEATURES=ROOT/"code"/"config"/"features.yaml"; MODELS=ROOT/"code"/"config"/"model_config.yaml"
SELECTED=RESULT_ROOT/"modeling"/"stage_3"/"selected_features.csv"; PARAMS=STAGE35/"final_refined_params.json"; A_SUMMARY=STAGE35/"final_tuning_summary.csv"; A_OOF={m:STAGE35/f"refined_{m}_oof_predictions.parquet" for m in ("logistic_regression","xgboost")}
LOCK=RESULT_ROOT/"modeling"/"sensitivity_sample_weight_weighted_class"; TUNE=LOCK/"tuning"; OUT=LOCK/"diagnostics"; LOCK_SUMMARY=LOCK/"locked_model_summary.csv"; LOCK_OOF={m:LOCK/f"locked_{m}_oof_predictions.parquet" for m in ("logistic_regression","xgboost")}; TUNE_OOF={m:TUNE/f"{m}_oof_predictions.parquet" for m in ("logistic_regression","xgboost")}
required=[DATASET,SPLIT,FEATURES,SELECTED,A_SUMMARY,*A_OOF.values(),LOCK_SUMMARY,*LOCK_OOF.values(),TUNE/"tuning_summary.csv",TUNE/"weighted_class_audit.csv",*TUNE_OOF.values()]; missing=[str(x) for x in required if not x.exists()]
if missing: raise FileNotFoundError("필요한 Train artifact가 없습니다:\n"+"\n".join(missing))
base=load_saved_global_train(DATASET,SPLIT,FEATURES); selected=load_selected_feature_names(SELECTED); weights=calculate_train_sample_weight(base.groups); row_count=base.groups.groupby(base.groups).transform("size")
assert len(selected)==25 and "n_prior_periods" not in base.X and weights.groupby(base.groups).sum().sub(1).abs().le(1e-12).all()


## 2. SAMPID row-count 분포·subgroup OOF


In [ ]:
def oof(path): return normalize_oof_columns(pd.read_parquet(path))
def rows(strategy, summary, paths, status, stage=None):
    s=pd.read_csv(summary)
    if stage is not None: s=s.loc[s.stage.eq(stage)]
    s=s.set_index("model"); result=[]
    for model,path in paths.items():
        frame=oof(path); metric=calculate_binary_metrics(frame.y_true,frame.y_probability)
        result.append({"strategy":strategy,"model":model,"feature_count":int(s.loc[model,"feature_count"]),"parameter_status":status,"cv_f1_mean":s.loc[model,"cv_f1_mean"],"cv_f1_std":s.loc[model,"cv_f1_std"],"oof_accuracy":metric["accuracy"],"oof_precision":metric["precision"],"oof_recall":metric["recall"],"oof_f1":metric["f1"],"oof_roc_auc":metric["roc_auc"],"oof_average_precision":metric["average_precision"],"predicted_positive_rate":frame.y_predicted.mean()})
    return pd.DataFrame(result)
def paired(labels, frames):
    out=[]
    for label,left,right in labels:
        row=paired_sampid_bootstrap_f1(frames[left],frames[right]); row.insert(0,"comparison",label); out.append(row)
    return pd.concat(out,ignore_index=True)
row_distribution=pd.DataFrame({"SAMPID":base.groups,"row_count":row_count,"sample_weight":weights,"y":base.y}).groupby("row_count").agg(unique_SAMPID=("SAMPID","nunique"),person_period_rows=("SAMPID","size"),average_sample_weight=("sample_weight","mean"),sample_weight_total=("sample_weight","sum"),positive_count=("y","sum"),positive_rate=("y","mean")).reset_index()
row_distribution["person_ratio"]=row_distribution.unique_SAMPID/row_distribution.unique_SAMPID.sum(); row_distribution["row_ratio"]=row_distribution.person_period_rows/row_distribution.person_period_rows.sum(); display(row_distribution)
groups=[]
for strategy,paths in [("A Baseline",A_OOF),("C revised tuned",TUNE_OOF)]:
 for model,path in paths.items():
  x=subgroup_oof_metrics(oof(path),row_count,group_name="row_count"); x.insert(0,"model",model); x.insert(0,"strategy",strategy); groups.append(x)
row_group_metrics=pd.concat(groups,ignore_index=True); display(row_group_metrics)


## 3. weighted class correction audit


In [ ]:
weighted_audit=pd.read_csv(TUNE/"weighted_class_audit.csv"); display(weighted_audit)
fig,ax=plt.subplots(); weighted_audit.pivot(index="fold",columns="model",values="weighted_neg_pos_ratio").plot(ax=ax); plt.show()
fig,ax=plt.subplots(); weighted_audit.query("model=='xgboost'").plot(x="fold",y="scale_pos_weight",marker="o",ax=ax); plt.show()


## 4. A / C locked / C tuned stability


In [ ]:
def oof(path): return normalize_oof_columns(pd.read_parquet(path))
def rows(strategy, summary, paths, status, stage=None):
    s=pd.read_csv(summary)
    if stage is not None: s=s.loc[s.stage.eq(stage)]
    s=s.set_index("model"); result=[]
    for model,path in paths.items():
        frame=oof(path); metric=calculate_binary_metrics(frame.y_true,frame.y_probability)
        result.append({"strategy":strategy,"model":model,"feature_count":int(s.loc[model,"feature_count"]),"parameter_status":status,"cv_f1_mean":s.loc[model,"cv_f1_mean"],"cv_f1_std":s.loc[model,"cv_f1_std"],"oof_accuracy":metric["accuracy"],"oof_precision":metric["precision"],"oof_recall":metric["recall"],"oof_f1":metric["f1"],"oof_roc_auc":metric["roc_auc"],"oof_average_precision":metric["average_precision"],"predicted_positive_rate":frame.y_predicted.mean()})
    return pd.DataFrame(result)
def paired(labels, frames):
    out=[]
    for label,left,right in labels:
        row=paired_sampid_bootstrap_f1(frames[left],frames[right]); row.insert(0,"comparison",label); out.append(row)
    return pd.concat(out,ignore_index=True)
tuned=pd.read_csv(TUNE/"tuning_summary.csv"); tuned["strategy"]="C revised tuned"; tuned["parameter_status"]="tuned"; a=rows("A Baseline",A_SUMMARY,A_OOF,"locked",stage="stage_3_5"); locked=rows("C revised locked",LOCK_SUMMARY,LOCK_OOF,"locked"); comparison=pd.concat([a,locked,tuned],ignore_index=True)
base_f1=comparison.query("strategy=='A Baseline'").set_index("model").oof_f1; locked_f1=comparison.query("strategy=='C revised locked'").set_index("model").oof_f1; comparison["delta_oof_f1_vs_A"]=comparison.apply(lambda r:r.oof_f1-base_f1[r.model],axis=1); comparison["locked_to_tuned_delta_f1"]=comparison.apply(lambda r:r.oof_f1-locked_f1[r.model],axis=1); display(comparison)
frames={"A "+m:oof(A_OOF[m]) for m in A_OOF}|{"C locked "+m:oof(LOCK_OOF[m]) for m in LOCK_OOF}|{"C tuned "+m:oof(TUNE_OOF[m]) for m in TUNE_OOF}; bootstrap=pd.concat([paired([("C locked - A","C locked "+m,"A "+m),("C tuned - A","C tuned "+m,"A "+m),("C tuned - C locked","C tuned "+m,"C locked "+m)],frames).assign(model=m) for m in ("logistic_regression","xgboost")],ignore_index=True); display(bootstrap)


## 5. 저장·최종 진단 표


In [ ]:
OUT.mkdir(parents=True,exist_ok=True); comparison.to_csv(OUT/"diagnostics_summary.csv",index=False); row_distribution.to_csv(OUT/"sampid_row_count_distribution.csv",index=False); row_group_metrics.to_csv(OUT/"row_count_group_metrics.csv",index=False); weighted_audit.to_csv(OUT/"weighted_class_audit_summary.csv",index=False); bootstrap.to_csv(OUT/"paired_bootstrap_oof.csv",index=False)
fold=pd.read_json(TUNE/"fold_f1.json").melt(var_name="model",value_name="f1").rename_axis("fold").reset_index(); fold.to_csv(OUT/"fold_comparison.csv",index=False)
fig,ax=plt.subplots(); row_distribution.plot.bar(x="row_count",y="positive_rate",ax=ax); plt.show(); fig,ax=plt.subplots(); row_group_metrics.pivot_table(index="row_count",columns=["strategy","model"],values="f1").plot.bar(ax=ax); plt.show(); display(comparison); display(row_distribution); display(bootstrap)
